# Structured Streaming

**Topics covered in this module:**
1. What is Structured Streaming and how does it work?
2. `spark.readStream` — Read a Delta table as a stream source
3. Streaming data manipulation con **SQL** (vía Streaming Temp View)
4. Streaming data persistence con `writeStream` + `Trigger.availableNow`
5. Streaming data manipulation con **Python** (PySpark directo)
6. Observe checkpoint effects by inserting live data
7. Exactly-once guarantees and idempotency


---
## ⚠️ Important note: Limitations of Databricks Free Edition / Community Edition

This notebook is adapted to run on **serverless computing** (the only option in Free Edition / Community Edition). 
Serverless computing imposes streaming restrictions that the book **does not mention** because it was written for traditional clusters.

### What CANNOT be done in serverless
- ❌ `trigger(processingTime="5 seconds")` → error `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED`
- ❌ Default continuous trigger → `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED` error
- ❌ `display()` on a streaming DataFrame without checkpoint → error `TEMP_CHECKPOINT_LOCATION_NOT_SUPPORTED`

### What CAN be done (what we will do)
- ✅ `trigger(availableNow=True)` — the only trigger supported in serverless
- ✅ Persist to the Delta table with `writeStream.toTable(...)`
- ✅ Query the destination table with `%sql` (it's already static, no checkpoint needed)

### Educational implication
Instead of "let the stream run and watch it update itself", the pattern will be:

1. Insert data into the source.
2. Launch an incremental batch with `availableNow=True`.
3. Query the destination table.
4. Repeat → verify that only the new data is processed (thanks to the checkpoint).

The concept being taught is **the same** (incremental, checkpoint, stateful, exactly-once); only the demonstration mechanism changes.


---
## 0. Setup

Same pattern as previous modules:
- Create catalog/schema/volume if they don't exist.
- Copy the `courses` data from S3 to the volume.
- Create the Delta table `courses`, which we will use as the **stream source**.
- Define a directory of **checkpoints** within the volume.

In [0]:
# Environment variables 
catalog  = "main"
schema   = "school"
volume   = "raw_data"

base_path        = f"/Volumes/{catalog}/{schema}/{volume}"
courses_path     = f"{base_path}/courses"
checkpoints_path = f"{base_path}/checkpoints"

print("Base path      :", base_path)
print("Courses path   :", courses_path)
print("Checkpoints    :", checkpoints_path)


Base path      : /Volumes/main/school/raw_data
Courses path   : /Volumes/main/school/raw_data/courses
Checkpoints    : /Volumes/main/school/raw_data/checkpoints


In [0]:
# Create catalog / schema / volume structure if it does not exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{volume}")
print("Structure ready")


Structure ready


In [0]:
# Copy courses data from S3 to the volume
s3_base = "s3://dalhussein-books/DEA-Book/datasets/school/v1"
dbutils.fs.cp(f"{s3_base}/courses-csv", courses_path, recurse=True)
print("Courses data copied")


Courses data copied


In [0]:
# Create Delta table 'courses' (stream source)
# We use 'overwrite' mode so the notebook can be re-executed from scratch.
courses_df = spark.read.csv(courses_path, header=True, inferSchema=True, sep=";")
courses_df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.courses")

print("Table 'courses' ready")
spark.sql(f"SELECT * FROM {catalog}.{schema}.courses").show()


Table 'courses' ready
+---------+--------------------+-----------+----------------+-----+
|course_id|               title| instructor|        category|price|
+---------+--------------------+-----------+----------------+-----+
|      C10|Database Design S...|   Julia S.|Computer Science|   44|
|      C11|Business Intellig...| Tiffany M.|Computer Science|   38|
|      C12|            Big Data| Bernard M.|Computer Science|   30|
|      C01|Data Structures a...|   Tracy N.|Computer Science|   49|
|      C02|JavaScript Design...|     Ali M.|Computer Science|   28|
|      C03|      Neural Network|    Adam R.|Computer Science|   35|
|      C07|    Machine Learning|  Andriy R.|Computer Science|   33|
|      C08|   Quantum Computing|   Chris N.|Computer Science|   41|
|      C09|Advanced Data Str...|  Pierre B.|Computer Science|   24|
|      C04|Robot Dynamics an...|    Mark G.|Computer Science|   20|
|      C05|  Python Programming| Luciano C.|Computer Science|   47|
|      C06|       Deep Lea

In [0]:
# Set default schema to use %sql without prefix
spark.sql(f"USE {catalog}.{schema}")
print(f"Using: {catalog}.{schema}")


Using: main.school


---
## 1. What is Structured Streaming?

The core idea is simple:

> **A stream is a table to which rows keep being added forever.**

You write a query as if it were against a static table. Spark takes care of:
- Detecting new data as it arrives (via Delta's transaction log).
- Executing your query only on the new data (micro-batch).
- Updating the result in the destination.
- Saving the progress to a **checkpoint** to recover from failures.

```
DELTA SOURCE (append-only)
        │
        │  New micro-batches
        ▼
  spark.readStream          ← stream subscription
        │
        │  Transformations (SQL or Python)
        ▼
  writeStream.toTable()     ← incremental persistence
        │
        ▼
  DETINATION DELTA TABLE
```

**Golden rule — append-only:**  
For a table to be a valid source for a stream, its data can only be *added*. Existing records cannot be modified or deleted.


---
## 2. `spark.readStream` — Create a Streaming DataFrame

`spark.readStream` **It doesn't read data yet**.
What it does is subscribe to the Delta table and return a **streaming DataFrame**:

an execution plan based on data that may not have arrived yet.

| | `spark.read` | `spark.readStream` |
|---|---|---|
| When does it read? | Once, when the action is executed | When the stream is launched |
|What does it return? | Static DataFrame | Streaming DataFrame |
|How do I write to it? | `.write` | `.writeStream` |

In [0]:
# Create the Streaming DataFrame
# This does NOT execute any read yet.
# It only registers that the source is streaming.

stream_df = spark.readStream.table("courses")

# Verify that Spark recognizes it as streaming
print("Is streaming?", stream_df.isStreaming)   # True
print("Stream schema:")
stream_df.printSchema()

Is streaming? True
Stream schema:
root
 |-- course_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- instructor: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)



Notice: `isStreaming = True`.

If we had used `spark.read.table("courses")`, it would be `False`.

This property is *inherited*: any transformation you apply to `stream_df` will also be streaming.

---
## 3. Manipulation with SQL — via Streaming Temp View

SQL does not have `readStream`. The bridge is a **Streaming Temporary View**:
a temporary view created *from* a streaming DataFrame.

```
spark.readStream  →  streaming DataFrame
                            │
             createOrReplaceTempView()
                            │
                            ▼
                  streaming temp view   ← SQL la puede referenciar
```

> ⚠️ **Important:** The view is only *defined* in SQL. The actual execution of the
> stream (read, write) **still requires PySpark** because that's where
> we configure the checkpoint and trigger.


In [0]:
# Register the streaming temporary view
stream_df.createOrReplaceTempView("courses_streaming_tmp_vw")
print("Streaming view registered: courses_streaming_tmp_vw")


Streaming view registered: courses_streaming_tmp_vw


In [0]:
%sql
-- Wrap the aggregation in another temp view
-- This CAN be done in %sql because CREATE VIEW is pure DDL:
-- it does not execute the stream, it only registers the definition.
-- The resulting view is also STREAMING (hereda de courses_streaming_tmp_vw).
CREATE OR REPLACE TEMP VIEW instructor_counts_tmp_vw AS (
  SELECT instructor, COUNT(course_id) AS total_courses
  FROM   courses_streaming_tmp_vw
  GROUP BY instructor
)


In [0]:
# Verify that the view is streaming
result_stream_df = spark.table("instructor_counts_tmp_vw")
print("Is streaming?", result_stream_df.isStreaming)  # True
result_stream_df.printSchema()


Is streaming? True
root
 |-- instructor: string (nullable = true)
 |-- total_courses: long (nullable = false)



---
## 4. Persist with `writeStream` + `Trigger.availableNow`

`Trigger.availableNow` processes **everything available** at that moment in one or more micro-batches and stops automatically upon completion.

It is the only viable trigger in Free Edition/Community Edition.

### Output modes

| Mode | Behavior | When to Use It |
---|---|---|
|`append` (default) | Only adds new rows | Raw reads, events, logs |
|`complete` | Rewrites the entire table each batch | Aggregations (`GROUP BY`) |

Since `instructor_counts_tmp_vw` has `GROUP BY`, we will use **`complete`**.

In [0]:
# Persist the stream result (SQL version)
ckpt_instructor = f"{checkpoints_path}/instructor_counts"

(
  result_stream_df.writeStream
    .trigger(availableNow=True)                     # only trigger supported in serverless
    .outputMode("complete")                          # rewrites destination table
    .option("checkpointLocation", ckpt_instructor)
    .toTable("instructor_counts")
    .awaitTermination()                              # wait until completion
)

print("Stream completed.")
print("Checkpoint at:", ckpt_instructor)


Stream completed.
Checkpoint at: /Volumes/main/school/raw_data/checkpoints/instructor_counts


In [0]:
%sql
-- Query the result (already a static Delta table)
-- Here we CAN use %sql because the destination table is static, not streaming.
SELECT * FROM instructor_counts
ORDER BY total_courses DESC, instructor


instructor,total_courses
Adam R.,1
Ali M.,1
Andriy R.,1
Bernard M.,1
Chris N.,1
François R.,1
Julia S.,1
Luciano C.,1
Mark G.,1
Pierre B.,1


---
## 5. See the stream in action — Insert live data 🔴

Since we can't leave a stream "running live" in a serverless environment, we simulate the arrival of new data using the following pattern:

```
INSERT into the source → run the incremental batch → query the table
```

The important thing: the **checkpoint** remembers which version of the `courses` table we reached last time. The next execution **only processes the new data**,
doesn't reprocess everything.

In [0]:
%sql
-- Batch 1: insert 3 new courses
INSERT INTO courses VALUES
  ('C16', 'Generative AI',     'Pierre B.',   'Computer Science', 25),
  ('C17', 'Embedded Systems',  'Julia S.',    'Computer Science', 30),
  ('C18', 'Virtual Reality',   'Bernard M.',  'Computer Science', 35)


num_affected_rows,num_inserted_rows
3,3


In [0]:
# Run the incremental batch
# Reuse result_stream_df, ckpt_instructor and the same writeStream.
# Thanks to the checkpoint, Spark knows where it left off and only processes new data.

(
  result_stream_df.writeStream
    .trigger(availableNow=True)
    .outputMode("complete")
    .option("checkpointLocation", ckpt_instructor)
    .toTable("instructor_counts")
    .awaitTermination()
)
print("Incremental batch completed.")


Incremental batch completed.


In [0]:
%sql
-- Verify the result: now should appear Pierre B., Julia S. y Bernard M.
SELECT * FROM instructor_counts
ORDER BY total_courses DESC, instructor

instructor,total_courses
Bernard M.,2
Julia S.,2
Pierre B.,2
Adam R.,1
Ali M.,1
Andriy R.,1
Chris N.,1
François R.,1
Luciano C.,1
Mark G.,1


In [0]:
%sql
-- Batch 2: 3 more courses
INSERT INTO courses VALUES
  ('C19', 'Compiler Design',    'Sophie B.', 'Computer Science', 25),
  ('C20', 'Signal Processing',  'Sam M.',    'Computer Science', 30),
  ('C21', 'Operating Systems',  'Mark H.',   'Computer Science', 35)


num_affected_rows,num_inserted_rows
3,3


In [0]:
# Run the incremental batch again
(
  result_stream_df.writeStream
    .trigger(availableNow=True)
    .outputMode("complete")
    .option("checkpointLocation", ckpt_instructor)
    .toTable("instructor_counts")
    .awaitTermination()
)
print("Incremental batch completed (batch 2).")


Incremental batch completed (batch 2).


In [0]:
%sql
SELECT * FROM instructor_counts
ORDER BY total_courses DESC, instructor


instructor,total_courses
Bernard M.,2
Julia S.,2
Pierre B.,2
Adam R.,1
Ali M.,1
Andriy R.,1
Chris N.,1
François R.,1
Luciano C.,1
Mark G.,1


---
## 6. Manipulation with Python (PySpark directo)

With Python, you do not need the temporary view workaround.
You apply transformations directly to the streaming DataFrame.

```
spark.readStream  →  streaming DataFrame
                            │
                   groupBy / agg / filter
                            │
                            ▼
                  output streaming DataFrame
                            │
                     writeStream.toTable()
                            │
                            ▼
                     Delta destination table
```


In [0]:
import pyspark.sql.functions as F

# stream_df is still the same streaming DataFrame from section 2
# Apply the same aggregation but in Python.
# stream_df is immutable: this creates a NEW streaming DataFrame.

output_stream_df = (
    stream_df
      .groupBy("instructor")
      .agg(F.count("course_id").alias("total_courses"))
)

print("Is streaming?", output_stream_df.isStreaming)  # True
output_stream_df.printSchema()


Is streaming? True
root
 |-- instructor: string (nullable = true)
 |-- total_courses: long (nullable = false)



In [0]:
# Persist with availableNow (pure Python) 
# IMPORTANT: we use a DIFFERENT checkpoint from section 4.
# Each stream needs its own checkpoint location.

ckpt_py = f"{checkpoints_path}/instructor_counts_py"

(
  output_stream_df.writeStream
    .trigger(availableNow=True)
    .outputMode("complete")
    .option("checkpointLocation", ckpt_py)
    .toTable("instructor_counts_py")
    .awaitTermination()
)
print("Python processing completed.")

Python processing completed.


In [0]:
%sql
SELECT * FROM instructor_counts_py
ORDER BY total_courses DESC, instructor


instructor,total_courses
Bernard M.,2
Julia S.,2
Pierre B.,2
Adam R.,1
Ali M.,1
Andriy R.,1
Chris N.,1
François R.,1
Luciano C.,1
Mark G.,1


---
## 7. Exactly-once guarantee: idempotency

If we re-run the incremental batch **without adding new data**, the stream
detects via checkpoint that it has already processed everything and **doesn't do any further work**.

This is the **exactly-once** guarantee of Structured Streaming in action:
each record is processed only once, no matter how many times you re-run.


In [0]:
# Re-run without new data
print("Running again without inserting any new data...")

(
  output_stream_df.writeStream
    .trigger(availableNow=True)
    .outputMode("complete")
    .option("checkpointLocation", ckpt_py)
    .toTable("instructor_counts_py")
    .awaitTermination()
)

print("Done. The table result is identical — Spark did not reprocess anything.")


Running again without inserting any new data...
Done. The table result is identical — Spark did not reprocess anything.


In [0]:
%sql
-- The result should be identical to section 6
SELECT * FROM instructor_counts_py
ORDER BY total_courses DESC, instructor


instructor,total_courses
Bernard M.,2
Julia S.,2
Pierre B.,2
Adam R.,1
Ali M.,1
Andriy R.,1
Chris N.,1
François R.,1
Luciano C.,1
Mark G.,1


---
## 8. Checkpoint: what does it store and why does it matter?

- **Offsets**: The up to which version of the Delta transaction log the stream reached.
- **State (state store)**: The accumulated stateful operations (like `COUNT`).
- **Write-ahead log**: Operations pending confirmation in the sink.

**Important rules:**
- ✅ Each stream needs its **own** checkpoint location.
- ✅ If the stream fails, restarting it with the same checkpoint will resume from where it left off.
- ❌ The checkpoint cannot be shared between two different streams.
- ❌ If you delete the checkpoint, the stream starts from scratch (reprocesses everything).


In [0]:
# Inspect the checkpoint
print(f"Contents of {ckpt_py}:")
for f in dbutils.fs.ls(ckpt_py):
    print(f"  {f.path}")


Contents of /Volumes/main/school/raw_data/checkpoints/instructor_counts_py:
  dbfs:/Volumes/main/school/raw_data/checkpoints/instructor_counts_py/commits/
  dbfs:/Volumes/main/school/raw_data/checkpoints/instructor_counts_py/metadata
  dbfs:/Volumes/main/school/raw_data/checkpoints/instructor_counts_py/offsets/
  dbfs:/Volumes/main/school/raw_data/checkpoints/instructor_counts_py/state/


The key files you will see:
- `offsets/` — which Delta source versions were processed by each batch
- `commits/` — which batches completed successfully
- `state/` — the state store with the accumulated counts
- `metadata` — general stream information


---
## 9. Complete flow summary

### SQL Version (sandwich Python → SQL → Python)

---
```python
# 1. READ (Python — required)
stream_df = spark.readStream.table("courses")
stream_df.createOrReplaceTempView("courses_streaming_tmp_vw")
```
---
```sql
-- 2. TRANSFORMATION (SQL — pure DDL, does not execute the stream)
CREATE OR REPLACE TEMP VIEW instructor_counts_tmp_vw AS (
  SELECT instructor, COUNT(course_id) AS total_courses
  FROM   courses_streaming_tmp_vw
  GROUP BY instructor
)
```
---
```python
# 3. WRITE (Python — required)
result_stream_df = spark.table("instructor_counts_tmp_vw")
(result_stream_df.writeStream
   .trigger(availableNow=True)
   .outputMode("complete")
   .option("checkpointLocation", ckpt)
   .toTable("instructor_counts")
   .awaitTermination())
```

### Pure Python Version (cleaner)

```python
import pyspark.sql.functions as F

stream_df = spark.readStream.table("courses")

output_df = (stream_df
             .groupBy("instructor")
             .agg(F.count("course_id").alias("total_courses")))

(output_df.writeStream
   .trigger(availableNow=True)
   .outputMode("complete")
   .option("checkpointLocation", ckpt)
   .toTable("instructor_counts_py")
   .awaitTermination())
```

### Key Concepts

| Concept | What it is |
|---|---|
|`spark.readStream` | Subscription to a streaming source (doesn't read yet) |
|`isStreaming = True` | Property inherited by all transformations |
|Streaming Temp View | Bridge to reference the stream from SQL |
|Stateful query | Query that needs to remember previous batches (`COUNT`, etc.) |
|`outputMode("complete")` | Rewrites the entire destination table every batch — only with aggregations |
|`outputMode("append")` | Only appends new rows — for raw reading |
|Checkpoint | Saves offsets and stream state for resumption |
|`availableNow=True` | Processes all new data and stops — only trigger supported in serverless environments |

### Triggers — What Works Where

| Trigger | Classic Clusters | Serverless / Free Edition |
|---|---|---|
|`processingTime="N sec"` | ✅ | ❌ `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED` |
|`availableNow=True` | ✅ | ✅ |
|`once=True` | ⚠️ deprecated | ⚠️ deprecated |


---
## Clean Up


In [0]:
def clean_up():
    # Stop any active stream (if any remain)
    for s in spark.streams.active:
        print(f"Stopping stream: {s.name}")
        s.stop()

    print("Dropping tables...")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.courses")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.instructor_counts")
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.instructor_counts_py")

    print("Deleting files from the volume...")
    dbutils.fs.rm(courses_path,     True)
    dbutils.fs.rm(checkpoints_path, True)

    print("Dropping schema...")
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")

    print("Done ✓")


In [0]:
# Uncomment to clean everything at the end of the lesson
clean_up()


Dropping tables...
Deleting files from the volume...
Dropping schema...
Done ✓
